---
title: "Marine's Lab 4"

format:
  html:
    embed-resources: true

execute:
  echo: true
---


# Lab 4: Data is Delicious

# Part 1. Data from unstructured websites (Option A)

This [website](https://tastesbetterfromscratch.com/category/meal-plan/) contains many weekly meal plans. Choose one that seems delicious to you. Scrape the weekly meal plan into to create a Pandas DataFrame with the following columns:

Day of the Week

Name of Recipe

Link to Recipe

Price of Recipe


In [1]:
# import packages
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
# tried the code provided in the description, did not work but this way
# did from PA #2!

import requests
response = requests.get(
    "https://tastesbetterfromscratch.com/meal-plan-203/",
    headers={"User-Agent": "Mozilla/5.0"})

In [3]:
# checking
response.status_code

200

In [4]:
soup = BeautifulSoup(response.content, "html.parser")

In [5]:
# after inspecting, noting that each paragraph/day is between <p> to </p>
paragraphs = soup.find_all('p')
len(paragraphs)

41

there's several paragraphs in soup so to narrow it down we inspect again for attributes of the info we want abt the day



```
# <p class="has-text-align-left wp-block-paragraph"><strong>Tuesday</strong>: <a href="https://tastesbetterfromscratch.com/lemon-chicken-orzo-soup/">Lemon Chicken Orzo Soup</a> $14.00</p>
```



In [6]:
# find how many paragraphs in soup have the class of the meal plan information
len(soup.find_all("p", attrs={
    "class": "has-text-align-left wp-block-paragraph"
}))

5

In [7]:
# finds the 5 paragraphs w the desired attributes containing the meal plan information
meals = soup.find_all("p", attrs={
    "class": "has-text-align-left wp-block-paragraph"
})
meals

[<p class="has-text-align-left wp-block-paragraph"><strong>Monday</strong>: <a href="https://tastesbetterfromscratch.com/italian-meatball-subs/">Meatball Subs</a> $18.18</p>,
 <p class="has-text-align-left wp-block-paragraph"><strong>Tuesday</strong>: <a href="https://tastesbetterfromscratch.com/lemon-chicken-orzo-soup/">Lemon Chicken Orzo Soup</a> $14.00</p>,
 <p class="has-text-align-left wp-block-paragraph"><strong>Wednesday: </strong><a href="https://tastesbetterfromscratch.com/baked-popcorn-chicken/">Popcorn Chicken</a> $9.97</p>,
 <p class="has-text-align-left wp-block-paragraph"><strong>Thursday:</strong> <a href="https://tastesbetterfromscratch.com/healthy-chicken-pasta-salad/">Chicken Pasta Salad</a> $12.72</p>,
 <p class="has-text-align-left wp-block-paragraph"><strong>Friday:</strong> <a href="https://tastesbetterfromscratch.com/baked-ham-swiss-sliders/">Ham &amp; Swiss Sliders</a> $12.70</p>]

In [8]:
# show just the first one, which should be Monday
meal = meals[0]
meal

<p class="has-text-align-left wp-block-paragraph"><strong>Monday</strong>: <a href="https://tastesbetterfromscratch.com/italian-meatball-subs/">Meatball Subs</a> $18.18</p>

In [9]:
# 1. Day of the Week - in <strong>
week_day = meal.find('strong').text.replace(':', '').strip()
week_day

'Monday'

In [10]:
# 2. Name of the Recipe - in <a>
recipe_name = meal.find('a').text.strip()
recipe_name

'Meatball Subs'

In [11]:
# 3. Link to Recipe - <a>
recipe_link = meal.find('a')['href']
recipe_link

'https://tastesbetterfromscratch.com/italian-meatball-subs/'

In [12]:
# 4. Price of Recipe - in <p>
# split after $ sign
recipe_price = meal.text.split('$')[1]
recipe_price

'18.18'

In [13]:
# empty list for alllll meal plan info
meal_plan = []

for meal in meals:
  # Day of the Week
  week_day = meal.find('strong').text.replace(':', '').strip()
  # Name of Recipe
  recipe_name = meal.find('a').text.strip()
  # Link to Recipe
  recipe_link = meal.find('a')['href']
  # Price of Recipe
  recipe_price = meal.text.split('$')[1]

  # Append this data.
  meal_plan.append({
      'Day of the Week': week_day,
      'Name of Recipe': recipe_name,
      'Link to Recipe': recipe_link,
      'Price of Recipe ($)': recipe_price
  })


In [14]:
# pandas dataframe
import pandas as pd
meal_plan_203 = pd.DataFrame(meal_plan)
meal_plan_203

,Day of the Week,Name of Recipe,Link to Recipe,Price of Recipe ($)
0,Monday,Meatball Subs,https://tastesbetterfromscratch.com/italian-me...,18.18
1,Tuesday,Lemon Chicken Orzo Soup,https://tastesbetterfromscratch.com/lemon-chic...,14.00
2,Wednesday,Popcorn Chicken,https://tastesbetterfromscratch.com/baked-popc...,9.97
3,Thursday,Chicken Pasta Salad,https://tastesbetterfromscratch.com/healthy-ch...,12.72
4,Friday,Ham & Swiss Sliders,https://tastesbetterfromscratch.com/baked-ham-...,12.70


--------------------------------------------------

# Part 2. Data from an API (Option A)

Using the Tasty API from the practice activity, search for recipes that resemble the “Monday” recipe in your meal plan. When seaching for recipes on Tasty, try some fuzzy matching; it doesn’t have to be an exact match. For example, if the Monday recipe is “biscuit chicken pot pie” you might search for “pot pie” or “biscuit chicken”. You may need to use some text editing tricks to convert the recipe name into a searchable string.

Your free Tasty API account only allows 500 queries per month. You should not need more than this, but if you do end up with a recipe that has a large number of matches, you only need to scrape the first 100 of them. It’s also fine if there are fewer than 100 matches.



In [15]:
# provided by 'code snippet' in rapidapi
import requests

# Tasty API
domain = "https://tasty.p.rapidapi.com"
endpoint = "recipes/list"
url = f"{domain}/{endpoint}"

# search for recipes similar to Monday's recipe
querystring = {"q": "Meatball Subs", "size": "20", "from": "0"}

headers = {
    "X-RapidAPI-Key": "ba53a17fc2msh286066cf48aee4ap18a05djsn17eef4b18c2f",
    "X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

response = requests.get(url, headers=headers, params=querystring)

monday_meatballs = response.json()
monday_meatballs

{'count': 5,
 'results': [{'nutrition_visibility': 'auto',
   'country': 'US',
   'instructions': [{'start_time': 0,
     'appliance': 'oven',
     'end_time': 0,
     'temperature': 375,
     'id': 1886,
     'position': 1,
     'display_text': 'Preheat oven to 375°F (190°C).'},
    {'start_time': 1000,
     'appliance': None,
     'end_time': 10117,
     'temperature': None,
     'id': 1887,
     'position': 2,
     'display_text': 'Cut each refrigerated biscuit into 8 pieces.  Place in a greased 9x13 inch (23x33 cm) baking dish.'},
    {'start_time': 11117,
     'appliance': None,
     'end_time': 16517,
     'temperature': None,
     'id': 1888,
     'position': 3,
     'display_text': 'Pour on the sauce, and stir to coat all sides of the biscuits.'},
    {'start_time': 17667,
     'appliance': None,
     'end_time': 20817,
     'temperature': None,
     'id': 1889,
     'position': 4,
     'display_text': 'Place the meatballs on top of the mixture.'},
    {'start_time': 21817,
   

In [16]:
# convert the recipes to a dataframe
df_meatballs = pd.json_normalize(monday_meatballs['results'])
df_meatballs

,nutrition_visibility,country,instructions,keywords,facebook_posts,language,seo_path,id,brand,slug,...,nutrition.calories,nutrition.sugar,show.name,show.id,total_time_tier.tier,total_time_tier.display_tier,brand.image_url,brand.name,brand.id,brand.slug
0,auto,US,"[{'start_time': 0, 'appliance': 'oven', 'end_t...",None,[],eng,"8757513,9295874,64453",303,NaN,meatball-sub-bake,...,1165,48,Tasty,17,under_45_minutes,Under 45 minutes,NaN,NaN,NaN,NaN
1,auto,ZZ,"[{'start_time': 1000, 'appliance': None, 'end_...","buzzfeed, cheese, comfort food, dinner, garlic...",[],und,"9295812,9299134,64501",2352,NaN,meatball-sub-boats,...,1156,7,Tasty,17,under_1_hour,Under 1 hour,NaN,NaN,NaN,NaN
2,auto,GB,"[{'start_time': 0, 'appliance': None, 'end_tim...","food, how to, meatball, party, party food, rec...",[],eng,"9295812,9299134,64501",1184,NaN,swedish-meatball-subs,...,422,11,Tasty International: Proper Tasty,20,under_1_hour,Under 1 hour,NaN,NaN,NaN,NaN
3,auto,US,"[{'start_time': 0, 'appliance': 'oven', 'end_t...",,[],eng,"9295813,64489,9299494",8675,NaN,chicken-parmesan-meatball-sub,...,549,5,Tasty,17,under_1_hour,Under 1 hour,NaN,NaN,NaN,NaN
4,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...","butter chicken meatball sub, butter chicken sa...",[],eng,"9295810,64462",9787,NaN,butter-chicken-meatball-sub,...,666,7,Tasty,17,under_2.5_hours,Under 2.5 hours,https://img.buzzfeed.com/tasty-app-user-assets...,Knorr,161.0,knorr


5x75
5 recipes
75 variables like time, nutrition, etc --> we should cut down on that

In [17]:
len(df_meatballs)
# double checking how many are actual meatball recipes

5

In [18]:
# checking all variables
df_meatballs.columns

Index(['nutrition_visibility', 'country', 'instructions', 'keywords',
       'facebook_posts', 'language', 'seo_path', 'id', 'brand', 'slug',
       'servings_noun_singular', 'show_id', 'prep_time_minutes',
       'is_subscriber_content', 'sections', 'brand_id', 'tags', 'is_app_only',
       'name', 'compilations', 'num_servings', 'buzz_id',
       'tips_and_ratings_enabled', 'aspect_ratio', 'created_at', 'description',
       'draft_status', 'inspired_by_url', 'thumbnail_url',
       'thumbnail_alt_text', 'total_time_minutes', 'video_url', 'updated_at',
       'credits', 'approved_at', 'is_one_top', 'servings_noun_plural',
       'renditions', 'beauty_url', 'is_shoppable', 'topics',
       'video_ad_content', 'seo_title', 'yields', 'original_video_url',
       'canonical_id', 'cook_time_minutes', 'promotion', 'video_id',
       'user_ratings.count_positive', 'user_ratings.score',
       'user_ratings.count_negative', 'price.total', 'price.updated_at',
       'price.portion', 'price.co

In [19]:
# show the names of the similar recipes
df_meatballs[['name']]

,name
0,Meatball Sub Bake
1,Meatball Sub Boats
2,Swedish Meatball Subs
3,Chicken Parmesan Meatball Sub
4,Butter Chicken Meatball Sub


The Monday Meatball sub from the original meal plan had some details we can see if we can match similar recipes from Tasty.

It was a sub/hoagie style roll, cheese, and tomato based sauce.

Serving of 5

70 minutes total

For a cost of $18.18

Price per serving was $3.60

In [20]:
df_meatballs['price_total_dollars'] = df_meatballs['price.total'] / 100
df_meatballs['price_portion_dollars'] = df_meatballs['price.portion'] / 100

In [21]:
# show information about the similar recipes from Tasty
df_meatballs[[
    'name',
    'num_servings',
    'total_time_minutes',
    'price_total_dollars',
    'price_portion_dollars',
    'user_ratings.score',
    'video_url'
]]

,name,num_servings,total_time_minutes,price_total_dollars,price_portion_dollars,user_ratings.score,video_url
0,Meatball Sub Bake,4,45,14.0,3.5,0.925344,https://vid.tasty.co/output/32066/low_14951250...
1,Meatball Sub Boats,4,55,22.0,5.5,0.960894,https://vid.tasty.co/output/48910/low_15035094...
2,Swedish Meatball Subs,12,50,34.5,3.0,0.931707,https://vid.tasty.co/output/31218/low_14943554...
3,Chicken Parmesan Meatball Sub,4,50,19.0,4.5,0.970000,https://vid.tasty.co/output/292745/hls24_16860...
4,Butter Chicken Meatball Sub,4,160,28.0,7.0,1.000000,https://vid.tasty.co/mediaconvert/output/40077...


Searched the Tasty API for recipes similar to the meal for Monday in my meal plan #203 which was Meatball sub. The search provided five recipes found, which are different types of meatball subs.
From the tables, there is similar format of information the recipe name, total time to make, the price, price per serving, how good the recipe is, and there were even video tutorials that I found interesting. The Meatball Sub Bake is the most similar in terms of servings (serves 4 instead of 5) and price per serving ( costs 3.50 vs 3.60) to the original recipe of Monday Meatball Sub.
